In [8]:
# ─── Cell 1: Load data ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

BASE = "/Users/aryanjungchhetri/Developers/6th sem/individual"

batting_df  = pd.read_csv(f"{BASE}/data/processed/smat_batting_raw.csv")
bowling_df  = pd.read_csv(f"{BASE}/data/processed/smat_bowling_raw.csv")

print(f"✅ SMAT Batting : {batting_df.shape}")
print(f"✅ SMAT Bowling : {bowling_df.shape}")



# ─── Cell 2: Batting position tracking ───────────────────────────────────────
# We need to go back to raw JSON to track batting order
import json

DATA_FOLDER = f"{BASE}/data/raw/Syed_Mushtaq_Ali_Trophy_Raw"
all_files   = [f for f in os.listdir(DATA_FOLDER) if f.endswith(".json")]

position_records = []

for filename in all_files:
    with open(os.path.join(DATA_FOLDER, filename), "r") as f:
        match = json.load(f)

    info  = match["info"]
    teams = info["teams"]

    for inning in match["innings"]:
        batting_team  = inning["team"]
        batting_order = []

        for over_data in inning["overs"]:
            for delivery in over_data["deliveries"]:
                batter = delivery["batter"]
                if batter not in batting_order:
                    batting_order.append(batter)
                non_striker = delivery["non_striker"]
                if non_striker not in batting_order:
                    batting_order.append(non_striker)

        for position, player in enumerate(batting_order, start=1):
            position_records.append({
                "player"   : player,
                "match_file": filename,
                "team"     : batting_team,
                "position" : position
            })

position_df = pd.DataFrame(position_records)
print(f"✅ Position records: {position_df.shape}")
print(position_df.head(10).to_string())







# ─── Cell 3: Aggregate batting position per player ───────────────────────────
position_agg = position_df.groupby("player")["position"].agg([
    "mean", "median", "min", "max",
    lambda x: x.mode()[0]   # most frequent position
]).reset_index()

position_agg.columns = [
    "player", "avg_position", "median_position",
    "highest_position", "lowest_position", "typical_position"
]

# Assign role based on typical position
def assign_position_role(pos):
    if pos <= 2:
        return "opener"
    elif pos <= 4:
        return "top_order"
    elif pos <= 6:
        return "middle_order"
    elif pos <= 8:
        return "lower_order"
    else:
        return "tailender"

position_agg["position_role"] = position_agg["typical_position"].apply(assign_position_role)

print("✅ Position aggregated")
print(position_agg.head(10).to_string())
print("\n=== POSITION ROLE DISTRIBUTION ===")
print(position_agg["position_role"].value_counts().to_string())




# ─── Cell 4: Full batting feature engineering ────────────────────────────────
def compute_batting_features(df):
    grp = pd.DataFrame()

    g = df.groupby("player")

    # ── Basic ──────────────────────────────────────────────────────────────
    grp["total_runs"]         = g["runs"].sum()
    grp["total_matches"]      = g["match_file"].nunique()
    grp["total_balls_faced"]  = g["balls_faced"].sum()
    grp["total_fours"]        = g["fours"].sum()
    grp["total_sixes"]        = g["sixes"].sum()
    grp["total_dismissed"]    = g["dismissed"].sum()

    # ── Averages ───────────────────────────────────────────────────────────
    grp["avg_runs"]           = g["runs"].mean().round(2)
    grp["highest_score"]      = g["runs"].max()
    grp["lowest_score"]       = g["runs"].min()

    # ── Not out rate ───────────────────────────────────────────────────────
    grp["not_out_rate"]       = (
        1 - (grp["total_dismissed"] / grp["total_matches"])
    ).round(3)

    # ── Duck rate ──────────────────────────────────────────────────────────
    grp["duck_count"]         = g.apply(
        lambda x: ((x["runs"] == 0) & (x["dismissed"] == 1)).sum()
    )
    grp["duck_rate"]          = (
        grp["duck_count"] / grp["total_matches"]
    ).round(3)

    # ── Consistency ────────────────────────────────────────────────────────
    grp["consistency_score"]  = g["runs"].std().fillna(0).round(2)
    grp["innings_above_30"]   = g.apply(lambda x: (x["runs"] >= 30).sum())
    grp["innings_above_50"]   = g.apply(lambda x: (x["runs"] >= 50).sum())
    grp["innings_above_30_pct"] = (
        grp["innings_above_30"] / grp["total_matches"]
    ).round(3)
    grp["innings_above_50_pct"] = (
        grp["innings_above_50"] / grp["total_matches"]
    ).round(3)

    # ── Strike rate ────────────────────────────────────────────────────────
    grp["strike_rate"]        = (
        grp["total_runs"] /
        grp["total_balls_faced"].replace(0, np.nan) * 100
    ).round(2)

    # ── Boundary & scoring style ───────────────────────────────────────────
    grp["boundary_rate"]      = (
        (grp["total_fours"] + grp["total_sixes"]) /
        grp["total_balls_faced"].replace(0, np.nan)
    ).round(3)

    grp["six_rate"]           = (
        grp["total_sixes"] /
        grp["total_balls_faced"].replace(0, np.nan)
    ).round(3)

    grp["four_rate"]          = (
        grp["total_fours"] /
        grp["total_balls_faced"].replace(0, np.nan)
    ).round(3)

    boundary_runs             = (grp["total_fours"] * 4) + (grp["total_sixes"] * 6)
    grp["boundary_runs_pct"]  = (
        boundary_runs /
        grp["total_runs"].replace(0, np.nan)
    ).round(3)

    grp["dot_ball_rate"]      = (
        g["dot_balls"].sum() /
        grp["total_balls_faced"].replace(0, np.nan)
    ).round(3)

    # ── Phase runs percentage ──────────────────────────────────────────────
    pp_runs  = g["pp_runs"].sum()
    mid_runs = g["mid_runs"].sum()
    dth_runs = g["death_runs"].sum()
    pp_balls  = g["pp_balls"].sum()
    mid_balls = g["mid_balls"].sum()
    dth_balls = g["death_balls"].sum()

    grp["pp_runs_pct"]        = (pp_runs  / grp["total_runs"].replace(0, np.nan)).round(3)
    grp["mid_runs_pct"]       = (mid_runs / grp["total_runs"].replace(0, np.nan)).round(3)
    grp["death_runs_pct"]     = (dth_runs / grp["total_runs"].replace(0, np.nan)).round(3)

    # ── Phase strike rates ─────────────────────────────────────────────────
    grp["pp_strike_rate"]     = (pp_runs  / pp_balls.replace(0, np.nan)  * 100).round(2)
    grp["mid_strike_rate"]    = (mid_runs / mid_balls.replace(0, np.nan) * 100).round(2)
    grp["death_strike_rate"]  = (dth_runs / dth_balls.replace(0, np.nan) * 100).round(2)

    # ── Dominant phase ─────────────────────────────────────────────────────
    phase_df = pd.DataFrame({
        "pp"   : grp["pp_strike_rate"].fillna(0),
        "mid"  : grp["mid_strike_rate"].fillna(0),
        "death": grp["death_strike_rate"].fillna(0)
    })
    grp["dominant_phase"]     = phase_df.idxmax(axis=1)

    # ── Win contribution ───────────────────────────────────────────────────
    win_mask  = df["team"] == df["winner"]
    loss_mask = df["team"] != df["winner"]

    grp["avg_runs_in_wins"]   = df[win_mask].groupby("player")["runs"].mean().round(2)
    grp["avg_runs_in_losses"] = df[loss_mask].groupby("player")["runs"].mean().round(2)

    # ── Team ───────────────────────────────────────────────────────────────
    grp["team"]               = g["team"].agg(lambda x: x.value_counts().index[0])

    return grp.reset_index()

batting_features = compute_batting_features(batting_df)
print(f"✅ Batting features: {batting_features.shape}")
print(batting_features.head(3).to_string())






# ─── Cell 5: Full bowling feature engineering ─────────────────────────────────
def compute_bowling_features(df):
    grp = pd.DataFrame()
    g   = df.groupby("player")

    # ── Basic ──────────────────────────────────────────────────────────────
    grp["total_wickets"]          = g["wickets"].sum()
    grp["total_matches_bowled"]   = g["match_file"].nunique()
    grp["total_balls_bowled"]     = g["balls_bowled"].sum()
    grp["total_runs_conceded"]    = g["runs_conceded"].sum()
    grp["total_wides"]            = g["wides"].sum()
    grp["total_noballs"]          = g["noballs"].sum()

    # ── Economy & averages ─────────────────────────────────────────────────
    grp["economy_rate"]           = (
        grp["total_runs_conceded"] /
        (grp["total_balls_bowled"].replace(0, np.nan) / 6)
    ).round(2)

    grp["bowling_average"]        = (
        grp["total_runs_conceded"] /
        grp["total_wickets"].replace(0, np.nan)
    ).round(2)

    grp["bowling_strike_rate"]    = (
        grp["total_balls_bowled"] /
        grp["total_wickets"].replace(0, np.nan)
    ).round(2)

    grp["avg_wickets_per_match"]  = (
        grp["total_wickets"] /
        grp["total_matches_bowled"]
    ).round(3)

    # ── Consistency ────────────────────────────────────────────────────────
    grp["bowling_consistency"]    = g["wickets"].std().fillna(0).round(2)

    grp["two_plus_hauls"]         = g.apply(lambda x: (x["wickets"] >= 2).sum())
    grp["two_plus_haul_rate"]     = (
        grp["two_plus_hauls"] /
        grp["total_matches_bowled"]
    ).round(3)

    # ── Accuracy ───────────────────────────────────────────────────────────
    grp["bowling_dot_rate"]       = (
        g["dot_balls"].sum() /
        grp["total_balls_bowled"].replace(0, np.nan)
    ).round(3)

    grp["wide_rate"]              = (
        grp["total_wides"] /
        grp["total_matches_bowled"]
    ).round(3)

    grp["noball_rate"]            = (
        grp["total_noballs"] /
        grp["total_matches_bowled"]
    ).round(3)

    # ── Phase economy ──────────────────────────────────────────────────────
    pp_runs   = g["pp_runs"].sum()
    mid_runs  = g["mid_runs"].sum()
    dth_runs  = g["death_runs"].sum()
    pp_balls  = g["pp_balls"].sum()
    mid_balls = g["mid_balls"].sum()
    dth_balls = g["death_balls"].sum()

    grp["pp_economy"]             = (pp_runs  / pp_balls.replace(0, np.nan)  * 6).round(2)
    grp["mid_economy"]            = (mid_runs / mid_balls.replace(0, np.nan) * 6).round(2)
    grp["death_economy"]          = (dth_runs / dth_balls.replace(0, np.nan) * 6).round(2)

    # ── Phase wicket rates ─────────────────────────────────────────────────
    # Need per-match phase wickets — approximate from economy patterns
    grp["pp_wicket_contribution"]    = (pp_balls  / grp["total_balls_bowled"].replace(0, np.nan)).round(3)
    grp["mid_wicket_contribution"]   = (mid_balls / grp["total_balls_bowled"].replace(0, np.nan)).round(3)
    grp["death_wicket_contribution"] = (dth_balls / grp["total_balls_bowled"].replace(0, np.nan)).round(3)

    # ── Dominant phase (lowest economy = best phase) ───────────────────────
    phase_eco = pd.DataFrame({
        "powerplay": grp["pp_economy"].fillna(999),
        "middle"   : grp["mid_economy"].fillna(999),
        "death"    : grp["death_economy"].fillna(999)
    })
    grp["dominant_phase"]         = phase_eco.idxmin(axis=1)

    # ── Bowling role label ─────────────────────────────────────────────────
    def assign_bowling_role(row):
        pp_contrib  = row["pp_wicket_contribution"]
        mid_contrib = row["mid_wicket_contribution"]
        dth_contrib = row["death_wicket_contribution"]

        if pp_contrib >= 0.4:
            return "powerplay_bowler"
        elif dth_contrib >= 0.4:
            return "death_bowler"
        elif mid_contrib >= 0.5:
            return "middle_overs_bowler"
        else:
            return "all_phases_bowler"

    grp["bowling_role"]           = grp.apply(assign_bowling_role, axis=1)

    # ── Win contribution ───────────────────────────────────────────────────
    win_mask  = df["team"] == df["winner"]
    loss_mask = df["team"] != df["winner"]

    grp["wickets_in_wins"]        = df[win_mask].groupby("player")["wickets"].mean().round(2)
    grp["wickets_in_losses"]      = df[loss_mask].groupby("player")["wickets"].mean().round(2)

    # ── Team ───────────────────────────────────────────────────────────────
    grp["team"]                   = g["team"].agg(lambda x: x.value_counts().index[0])

    return grp.reset_index()

bowling_features = compute_bowling_features(bowling_df)
print(f"✅ Bowling features: {bowling_features.shape}")
print(bowling_features.head(3).to_string())




# ─── Cell 6: Merge batting position into batting features ─────────────────────
batting_features = batting_features.merge(
    position_agg[["player", "typical_position", "avg_position", "position_role"]],
    on="player",
    how="left"
)

# Fill missing positions
batting_features["typical_position"] = batting_features["typical_position"].fillna(0)
batting_features["position_role"]    = batting_features["position_role"].fillna("unknown")

print("✅ Position merged into batting features")
print(batting_features[["player", "typical_position", "position_role", "strike_rate", "dominant_phase"]].head(10).to_string())





# ─── Cell 7: Assign batting role label ────────────────────────────────────────
def assign_batting_role(row):
    pos   = row["typical_position"]
    pp_sr = row["pp_strike_rate"]   if not pd.isna(row["pp_strike_rate"])    else 0
    dt_sr = row["death_strike_rate"] if not pd.isna(row["death_strike_rate"]) else 0
    sr    = row["strike_rate"]       if not pd.isna(row["strike_rate"])       else 0
    avg   = row["avg_runs"]

    if pos <= 2 and pp_sr >= 120:
        return "aggressive_opener"
    elif pos <= 2:
        return "anchor_opener"
    elif pos <= 4 and avg >= 25:
        return "top_order_anchor"
    elif pos <= 4 and sr >= 140:
        return "top_order_aggressor"
    elif pos <= 6 and dt_sr >= 150:
        return "finisher"
    elif pos <= 6:
        return "middle_order"
    elif sr >= 150:
        return "lower_order_hitter"
    else:
        return "tailender"

batting_features["batting_role"] = batting_features.apply(assign_batting_role, axis=1)

print("✅ Batting role assigned")
print("\n=== BATTING ROLE DISTRIBUTION ===")
print(batting_features["batting_role"].value_counts().to_string())





# ─── Cell 8: Filter minimum matches & save ────────────────────────────────────
MIN_BAT_MATCHES  = 3
MIN_BOWL_MATCHES = 3

batting_final = batting_features[
    batting_features["total_matches"] >= MIN_BAT_MATCHES
].copy()

bowling_final = bowling_features[
    bowling_features["total_matches_bowled"] >= MIN_BOWL_MATCHES
].copy()

print(f"✅ Batting : {len(batting_final)} players (was {len(batting_features)})")
print(f"✅ Bowling : {len(bowling_final)} players (was {len(bowling_features)})")

# Save
batting_final.to_csv(f"{BASE}/data/processed/smat_batting_features.csv", index=False)
bowling_final.to_csv(f"{BASE}/data/processed/smat_bowling_features.csv", index=False)

print("\n✅ Saved:")
print("   → data/processed/smat_batting_features.csv")
print("   → data/processed/smat_bowling_features.csv")





# ─── Cell 9: Sanity checks ────────────────────────────────────────────────────
print("=== TOP 10 BATTERS BY STRIKE RATE ===")
print(batting_final.nlargest(10, "strike_rate")[
    ["player", "total_matches", "avg_runs", "strike_rate",
     "boundary_rate", "batting_role", "dominant_phase"]
].to_string())

print("\n=== TOP 10 AGGRESSIVE OPENERS ===")
openers = batting_final[batting_final["batting_role"] == "aggressive_opener"]
print(openers.nlargest(10, "strike_rate")[
    ["player", "total_matches", "avg_runs", "strike_rate", "pp_strike_rate"]
].to_string())

print("\n=== TOP 10 FINISHERS ===")
finishers = batting_final[batting_final["batting_role"] == "finisher"]
print(finishers.nlargest(10, "death_strike_rate")[
    ["player", "total_matches", "avg_runs", "death_strike_rate", "six_rate"]
].to_string())

print("\n=== TOP 10 BOWLERS BY ECONOMY ===")
print(bowling_final.nsmallest(10, "economy_rate")[
    ["player", "total_matches_bowled", "economy_rate",
     "total_wickets", "bowling_role", "dominant_phase"]
].to_string())

print("\n=== TOP 10 DEATH BOWLERS ===")
death_bowlers = bowling_final[bowling_final["bowling_role"] == "death_bowler"]
print(death_bowlers.nsmallest(10, "death_economy")[
    ["player", "total_matches_bowled", "death_economy",
     "total_wickets", "avg_wickets_per_match"]
].to_string())

print("\n=== TOP 10 POWERPLAY BOWLERS ===")
pp_bowlers = bowling_final[bowling_final["bowling_role"] == "powerplay_bowler"]
print(pp_bowlers.nsmallest(10, "pp_economy")[
    ["player", "total_matches_bowled", "pp_economy",
     "total_wickets", "bowling_dot_rate"]
].to_string())









# ─── Full sanity check without truncation ────────────────────────────────────
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

print("=== BATTING FEATURES SHAPE ===")
print(f"Total players : {len(batting_final)}")
print(f"Total columns : {len(batting_final.columns)}")
print(f"Columns : {list(batting_final.columns)}")

print("\n=== BATTING ROLE DISTRIBUTION ===")
print(batting_final["batting_role"].value_counts().to_string())

print("\n=== POSITION ROLE DISTRIBUTION ===")
print(batting_final["position_role"].value_counts().to_string())

print("\n=== DOMINANT PHASE DISTRIBUTION (BATTING) ===")
print(batting_final["dominant_phase"].value_counts().to_string())

print("\n=== BOWLING ROLE DISTRIBUTION ===")
print(bowling_final["bowling_role"].value_counts().to_string())

print("\n=== DOMINANT PHASE DISTRIBUTION (BOWLING) ===")
print(bowling_final["dominant_phase"].value_counts().to_string())

print("\n=== TOP 10 BATTERS BY AVG RUNS ===")
print(batting_final.nlargest(10, "avg_runs")[
    ["player", "total_matches", "avg_runs", "strike_rate",
     "batting_role", "dominant_phase", "typical_position"]
].to_string())

print("\n=== TOP 10 BOWLERS BY ECONOMY ===")
print(bowling_final.nsmallest(10, "economy_rate")[
    ["player", "total_matches_bowled", "economy_rate",
     "total_wickets", "bowling_role", "dominant_phase"]
].to_string())

print("\n=== TOP 10 FINISHERS (death_strike_rate) ===")
finishers = batting_final[batting_final["batting_role"] == "finisher"]
print(f"Total finishers: {len(finishers)}")
print(finishers.nlargest(10, "death_strike_rate")[
    ["player", "total_matches", "avg_runs",
     "death_strike_rate", "six_rate", "typical_position"]
].to_string())

print("\n=== TOP 10 AGGRESSIVE OPENERS ===")
openers = batting_final[batting_final["batting_role"] == "aggressive_opener"]
print(f"Total aggressive openers: {len(openers)}")
print(openers.nlargest(10, "pp_strike_rate")[
    ["player", "total_matches", "avg_runs",
     "pp_strike_rate", "strike_rate", "boundary_rate"]
].to_string())

print("\n=== TOP 10 DEATH BOWLERS ===")
death = bowling_final[bowling_final["bowling_role"] == "death_bowler"]
print(f"Total death bowlers: {len(death)}")
print(death.nsmallest(10, "death_economy")[
    ["player", "total_matches_bowled", "death_economy",
     "total_wickets", "two_plus_haul_rate"]
].to_string())

print("\n=== TOP 10 POWERPLAY BOWLERS ===")
pp = bowling_final[bowling_final["bowling_role"] == "powerplay_bowler"]
print(f"Total powerplay bowlers: {len(pp)}")
print(pp.nsmallest(10, "pp_economy")[
    ["player", "total_matches_bowled", "pp_economy",
     "total_wickets", "bowling_dot_rate"]
].to_string())

✅ SMAT Batting : (10569, 20)
✅ SMAT Bowling : (8278, 19)
✅ Position records: (10765, 4)
           player    match_file    team  position
0      KH Devdhar  1244407.json  Baroda         1
1       NA Rathva  1244407.json  Baroda         2
2  Vishnu Solanki  1244407.json  Baroda         3
3        SK Patel  1244407.json  Baroda         4
4     Bhanu Pania  1244407.json  Baroda         5
5       AV Rajput  1244407.json  Baroda         6
6       KR Kakade  1244407.json  Baroda         7
7         A Sheth  1244407.json  Baroda         8
8        BA Bhatt  1244407.json  Baroda         9
9       BA Pathan  1244407.json  Baroda        10
✅ Position aggregated
           player  avg_position  median_position  highest_position  lowest_position  typical_position position_role
0  A Aravinddaraj     11.000000             11.0                11               11                11     tailender
1  A Ashish Reddy      5.583333              6.0                 3                7                 6  middl